# 1. Dataset preparation

Use cropped image/TXT pairs. Your data stays in ignored local directories. Edit `configs/alpr.yaml` before starting.

Current training/evaluation policy: **literal-string exact only**, including spaces, symbols and case; earliest tied checkpoint wins. TXT permits one framing LF/CRLF only. Old training-selection state is incompatible with resume; historical reference cells retain their original scope. See [literal text policy](../docs/literal_text_policy.md).

## Model selection

Read [0_model_options.ipynb](0_model_options.ipynb) for Base, Tiny and the custom Fast Tiny recipe. Choose `configs/alpr_base.yaml`, `configs/alpr_tiny.yaml`, or `configs/alpr_fast_tiny.yaml` via `CONFIG_PATH`. Keep that choice consistent across notebooks and use separate output folders.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "parseq_ocr").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "parseq_ocr").is_dir():
    raise RuntimeError("Open this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))
from parseq_ocr.config import load_config
# Dummy presets: configs/dummy_base.yaml, dummy_tiny.yaml, dummy_fast_tiny.yaml.
# Real-data presets: configs/alpr_base.yaml, alpr_tiny.yaml, alpr_fast_tiny.yaml.
# PARSEQ_CONFIG overrides this default when it is set.
CONFIG_PATH = os.environ.get("PARSEQ_CONFIG", "configs/dummy_tiny.yaml")
config = load_config(CONFIG_PATH)
print("Config:", CONFIG_PATH)
print("Model:", config["model"], "input:", config["image_size"])
print("Artifacts:", config["output_dir"])


## Input format

Each crop has a UTF-8 label with the same stem. Supply `dataset/train` and `dataset/val`, or label crops in the optional annotator and let the next cell create a label-disjoint split.

```text
dataset/
  train/example.jpg
  train/example.txt
  val/another.jpg
  val/another.txt
```

The default ALPR normalization uppercases text and removes whitespace. It preserves punctuation; no country-format correction is used. The released model supports its original 94-character alphabet and up to 25 characters for standard presets (12 by default for Fast Tiny). Unsupported labels raise an error rather than being silently changed.

In [ ]:
from parseq_ocr.data import scan_pairs, audit_splits
import subprocess

# Generate only bundled synthetic workflow fixtures; no annotation conversion or downloads.
if config["dataset_root"] == ROOT / "dummy_data" / "plates":
    subprocess.run([sys.executable, str(ROOT / "tools" / "make_dummy_data.py")], check=True)
training = scan_pairs(config["dataset_root"] / "train", config["normalization"])
validation = scan_pairs(config["dataset_root"] / "val", config["normalization"])
audit = audit_splits(training, validation)
print(audit)
if audit["pixel_duplicates_across_splits"]:
    raise ValueError("TRAIN/validation pixels overlap; fix dataset before training")


In [ ]:
from parseq_ocr.data import split_holdout
from parseq_ocr.models import create_system
from parseq_ocr.data import validate_labels

system = create_system(config, pretrained=False)
validate_labels(training + validation, system.tokenizer, config.get("max_label_length", 25))
fit, holdout = split_holdout(training, config["holdout_fraction"], config["split_seed"])
print({"fit_images": len(fit), "internal_holdout_images": len(holdout)})
del system

The internal holdout uses training images only and groups identical label strings. Keep the supplied validation split and any external online dataset out of epoch selection. Exact pixel matching does not rule out all near-duplicates or unknown pretraining exposure.